# Shmoo Benchmarking for DSP


In [1]:
# Equipment setup

import pyvisa
import serial
import shlex
import subprocess
import time
import struct
import matplotlib.pyplot as plt
import numpy as np
from openocd import OpenOcdTclRpc
import logging
import traceback
LOGGER = logging.getLogger(__name__)
logging.basicConfig(filename='runs.log', level=logging.DEBUG)

from pyftdi.ftdi import Ftdi, UsbTools
from pyftdi.gpio import *
from time import sleep

import config

In [2]:
# Connect the SMU
rm = pyvisa.ResourceManager("@py")

smu = rm.open_resource('TCPIP::169.254.58.10::gpib0,13::INSTR')

smu_verification = smu.query("*IDN?")
print("SourceMeter:", smu_verification)
print("SMU Buffer Size", smu.query("print(maxNumber)"))

SourceMeter: Keithley Instruments Inc., Model 2602A, 1264983, 2.1.5

SMU Buffer Size 6.99010e+04



In [3]:
init_current_limit = "500e-3"
init_voltage_limit = "0.85"

def smu_set_limits(voltage=init_voltage_limit, current=init_current_limit):
    smu.write("smub.source.func = smub.OUTPUT_DCVOLTS")
    #$ smu.write("smub.source.autorangev = smub.AUTORANGE_ON")
    smu.write(f"smub.source.limitv = {init_voltage_limit}")
    smu.write(f"smub.source.levelv = {init_voltage_limit}")
    smu.write(f"smub.source.rangev = {init_voltage_limit}")

def smu_init():
    # Initialize SMU
    smu.write("smub.reset()")
    smu.write(f"smub.measure.rangei = {init_current_limit}")
    smu.write(f"smub.source.limiti = {init_current_limit}")
    smu_set_limits()
    # Clear and reset buffers
    smu.write("smub.nvbuffer1.clear()")
    smu.write("smub.nvbuffer1.collecttimestamps = 1")
    smu.write("smub.nvbuffer1.collectsourcevalues = 1")
    smu.write("smub.nvbuffer2.clear()")
    smu.write("smub.nvbuffer2.collecttimestamps = 1")
    smu.write("smub.nvbuffer2.collectsourcevalues = 1")

    # Capture count / delay
    smu.write("smub.measure.count = 700")
    smu.write("smub.measure.nplc = 0.1")
    smu_clear_buffer()

def smu_enable():
    smu.write("smub.source.output = smub.OUTPUT_ON")

def smu_disable():
    smu.write("smub.source.output = smub.OUTPUT_OFF")

def smu_clear_buffer(idx=1, collect_src_vals=True, collect_timestamps=True):
    """
    Clear and reset buffer collection settings.
    """
    collect_src_vals = '1' if collect_src_vals else '0'
    collect_timestamps = '1' if collect_timestamps else '0'
    smu.write(f"smub.nvbuffer{idx}.clear()")
    smu.write(f"smub.nvbuffer{idx}.collecttimestamps = 1")
    smu.write(f"smub.nvbuffer{idx}.collectsourcevalues = 1")
    smu.write(f"smub.nvbuffer{idx}.clear()")
    smu.write(f"smub.nvbuffer{idx}.collecttimestamps = 1")
    smu.write(f"smub.nvbuffer{idx}.collectsourcevalues = 1")

def smu_measure_p(count=700, nplc=0.1, buffer=1):
    """
    Measures power and returns a space-delimited list in the following order:
    Measurement 1, Timestamp 1, Voltage Source 1, Measurement 2, ...

    See Series 2600A System SourceMeter ® Instruments User’s Manual for more info.

    :param count: Number of measurements to take.
    :param nplc: Frequency of measurements to take. 0.01 is the absolute
                 minimum with very high noise.
    :param buffer: 1 or 2 depending on the buffer you want to use.
        
    """
    # Capture count / delay
    smu.write(f"smub.measure.count = {count}")
    smu.write(f"smub.measure.nplc = {nplc}")
    smu.write(f"smub.measure.p(smub.nvbuffer{buffer})")

smu_init()


In [4]:
# PLL Reference Clock (MHz)
pll_ref_clock = 50

bmark_name = 'shmoo_bench'
cmake_name = 'shmoo_bench'

In [8]:
# def update_freq(freq_mhz, target_name):
#     # freq_mhz = Frequency in MHz
#     # target_name = Executable name
#     with open("../../platform/dsp24/freq.h", "w") as f:
#         ratio = freq_mhz // pll_ref_clock
#         print(f'Creating header for {freq_mhz} MHz and (mult ratio of {ratio} using PLL reference of {pll_ref_clock} MHz)')
#         f.write('#ifndef __FREQ_H\n#define __FREQ_H\n')
#         f.write(f'#define MTIME_FREQ     {freq_mhz}000000\n')
#         f.write(f'#define SYS_CLK_FREQ   {freq_mhz}000000\n')
#         f.write(f'#define SHMOO_PLL_RATIO   {ratio}\n')
#         f.write("#endif")


#     with open("../CMakeLists.txt", "w") as f:
#         f.write(f'''add_executable({target_name}
#   dsp_conv_bench/src/main.c
# )\n''')
#         f.write(f'target_include_directories({target_name} PUBLIC dsp_conv_bench/include)\n\n')
#         f.write(f'target_link_libraries({target_name} PRIVATE \n')
#         f.write('  -L${CMAKE_BINARY_DIR}/glossy -Wl,--whole-archive glossy -Wl,--no-whole-archive)\n\n')

#         # Didnt work: f.write(f'set_target_properties({exec_name} PROPERTIES OUTPUT_NAME "{target_name}.elf")')

In [9]:
# # Precompiles all .elf files for future benchmarking.
# # Should only need to run this once.
# cur_clk = start_clk
# while cur_clk <= end_clk:
#     target_name = f'{cmake_name}__{cur_clk}'
#     update_freq(cur_clk, target_name)
#     # cmake_arg_soc = shlex.split(f"/tools/C/ee290-fa24-2/.conda-env/bin/cmake -S ./ -B ./build_shmoo/ -D CMAKE_TOOLCHAIN_FILE=./riscv-gcc.cmake -DCHIP=dsp24")
#     cmake_arg_shmoo = shlex.split(f"make build TARGET={target_name} CHIP=dsp24")
#     #subprocess.run(cmake_arg_soc, cwd="../../")
#     subprocess.run(cmake_arg_shmoo, cwd="../../")
    
#     cur_clk += step_clk

In [6]:
Ftdi.show_devices()


Available interfaces:
  ftdi://ftdi:2232:1:15/1  (Dual RS232-HS)
  ftdi://ftdi:2232:1:15/2  (Dual RS232-HS)



In [14]:
# def reset_chip():
#     gcont = GpioMpsseController()
#     gcont.configure("ftdi://ftdi:2232:1:3/1", direction=0x0100, frequency=10e6)
#     port = gcont.get_gpio()
#     port.write(0x00)
#     sleep(.5)
#     port.write(0x100)
#     gcont.close(freeze=1)

In [15]:

# gpib_cmd = '''errorqueue.clear()
# testFinished = 0
# display.clear()
# display.setcursor(1, 1)
# display.settext("start")
# smub.source.output = smub.OUTPUT_ON
# while (testFinished == 0) do smub.measure.p() end
# smub.source.output = smub.OUTPUT_OFF
# display.settext(" end")
# '''
# smu.write(gpib_cmd)

In [16]:
# smu.write('display.clear()')
# smu.write('display.setcursor(1, 1)')
# smu.write('display.settext("dying inside yay :D")')

In [5]:
#smu_enable()

def ftdi_reset():
    devices = UsbTools.build_dev_strings('ftdi:///?', Ftdi.VENDOR_IDS, Ftdi.PRODUCT_IDS, Ftdi.list_devices())
    if not devices:
        raise Exception("No FTDI device found. :(")
    gcont = GpioMpsseController()
    gcont.configure(devices[0][0], direction=0x0100, frequency=10e6)
    port = gcont.get_gpio()
    port.write(0x00)
    sleep(.5)
    gcont.close()

def ocd_start():
    openocd_args = shlex.split("make ocd CHIP=dsp24")
    return subprocess.Popen(openocd_args, cwd="/tools/C/angle/shmoo")

def ocd_cmd(ocd, cmd):
    LOGGER.debug(f"OpenOCD | {cmd} | {ocd.run(cmd)}")

def int_to_32bit_hex(integer):
    return '0x{:08x}'.format(integer & 0xFFFFFFFF)

def ocd_upload_pgm(elf, autorun=True, freq_mhz=50):
    # Upload the program
    with OpenOcdTclRpc() as openocd:
        #ocd_cmd(openocd, 'reset halt')
        #ocd_cmd(openocd, 'set_reg {pc 0x80000000}')
        ocd_cmd(openocd, f'load_image {elf} 0x0 elf')
        freq_ratio = freq_mhz // pll_ref_clock

        #if freq_ratio > 1:
        #    print("Test")
            # Clock Selector = 0 (50MHz Slow Clock)
            # ocd_cmd(openocd, 'read_memory 0x130000 32 1')
            # ocd_cmd(openocd, 'write_memory 0x130000 32 {0x00000000}')
            
            # PLL Settings
            
            # ratio = int_to_32bit_hex(freq_ratio)
            # ocd_cmd(openocd, 'write_memory 0x140060 32 {0x00000000}') # PLLEN = 0;
            # ocd_cmd(openocd, 'write_memory 0x140074 32 {0x00000001}') # MDIV_RATIO = 1;
            # ocd_cmd(openocd, f'write_memory 0x14006C 32 {{{ratio}}}') # RATIO = ratio;
            # ocd_cmd(openocd, 'write_memory 0x140070 32 {0x00000000}') # FRACTION = 0;
            # ocd_cmd(openocd, 'write_memory 0x140078 32 {0x00000001}') # ZDIV0_RATIO = 1;
            # ocd_cmd(openocd, 'write_memory 0x140080 32 {0x00000001}') # ZDIV1_RATIO = 1;
            # ocd_cmd(openocd, 'write_memory 0x140064 32 {0x00000001}') # LDO_ENABLE = 1;
            # ocd_cmd(openocd, 'write_memory 0x140060 32 {0x00000001}') # PLLEN = 1;
            # ocd_cmd(openocd, 'write_memory 0x14005C 32 {0x00000001}') # POWERGOOD_VNN = 1;
            # ocd_cmd(openocd, 'write_memory 0x140100 32 {0x00000001}') # PLLFWEN_B = 1;
            
            # ocd_cmd(openocd, 'read_memory 0x140060 32 1') # PLLEN = 0;
            # ocd_cmd(openocd, 'read_memory 0x140074 32 1') # MDIV_RATIO = 1;
            # ocd_cmd(openocd, 'read_memory 0x14006C 32 1') # RATIO = ratio;
            # ocd_cmd(openocd, 'read_memory 0x140070 32 1') # FRACTION = 0;
            # ocd_cmd(openocd, 'read_memory 0x140078 32 1') # ZDIV0_RATIO = 1;
            # ocd_cmd(openocd, 'read_memory 0x140080 32 1') # ZDIV1_RATIO = 1;
            # ocd_cmd(openocd, 'read_memory 0x140064 32 1') # LDO_ENABLE = 1;
            # ocd_cmd(openocd, 'read_memory 0x140060 32 1') # PLLEN = 1;
            # ocd_cmd(openocd, 'read_memory 0x14005C 32 1') # POWERGOOD_VNN = 1;
            # ocd_cmd(openocd, 'read_memory 0x140100 32 1') # PLLFWEN_B = 1;

            # Clock Selector = 1 (PLL)
            #ocd_cmd(openocd, 'read_memory 0x130000 32 1')
            #ocd_cmd(openocd, 'write_memory 0x130000 32 {0x00000001}') # Clock Selector = 1;

        # Set UART divisor
        # b = inputfreq/baud-1
        uart_divisor = ((freq_mhz*1000000) // 115200) - 1
        uart_divisor_hex = int_to_32bit_hex(uart_divisor)
        #ocd_cmd(openocd, f'write_memory 0x10020000 32 {{{uart_divisor_hex}}}')
        ocd_cmd(openocd, 'resume 0x80000000')
        LOGGER.info(f'Completed setup with UART clock divisor set by chip and running freq of {freq_mhz} MHz')
        LOGGER.info(f'We should expect a UART clock divisor of {uart_divisor} ({uart_divisor_hex})')


# ocd_upload_pgm("shmoo_bench", True, freq_mhz=100)


# write_memory 0x130000 32 {0x00000000}
# write_memory 0x140060 32 {0x00000000}
# write_memory 0x140074 32 {0x00000001}
# write_memory 0x14006C 32 {0x00000002}
# write_memory 0x140070 32 {0x00000000}
# write_memory 0x140078 32 {0x00000001}
# write_memory 0x140080 32 {0x00000001}
# write_memory 0x140064 32 {0x00000001}
# write_memory 0x140060 32 {0x00000001}
# write_memory 0x14005C 32 {0x00000001}
# write_memory 0x140100 32 {0x00000001}



In [10]:
ftdi_reset()
logging.basicConfig(filename='runs.log', level=logging.DEBUG)

class SerialDebug(serial.Serial):
    def __init__(self, *args, **kwargs):
        self.logger = logging.getLogger(__name__)
        super().__init__(*args, **kwargs)
        self.logger.debug(f"Serial port {self.port} initialized with settings: {self.get_settings()}")

    def read(self, size=1):
        data = super().read(size)
        self.logger.debug(f"Serial Received: {data!r}")
        return data

    def write(self, data):
        self.logger.debug(f"Serial Sent: {data!r}")
        return super().write(data)

    def close(self):
        self.logger.debug(f"Closing serial port {self.port}")
        super().close()

possible_tty = range(1, 3)
for i in possible_tty:
    try:
        ser = SerialDebug(f'/dev/ttyUSB{i}', 115200)
        LOGGER.debug(f'Initialized serial connection via {ser.name}')
        break
    except:
        LOGGER.debug(f'Could not initialize /dev/ttyUSB{i}.')

In [14]:
'''
OCD sets PLL MMIO over JTAG
    [CLOCK_SELECTOR->SEL = 0;
        PLL->PLLEN = 0;
        PLL->MDIV_RATIO = 1;
        PLL->RATIO = 10;  // 500MHz
        PLL->FRACTION = 0;
        PLL->ZDIV0_RATIO = 1;
        PLL->ZDIV1_RATIO = 1;
        PLL->LDO_ENABLE = 1;
        PLL->PLLEN = 1;
        PLL->POWERGOOD_VNN = 1;
        PLL->PLLFWEN_B = 1;
        CLOCK_SELECTOR->SEL = 1; // Switch to PLL]
OCD sets baud rate divisor over JTAG for UART
OCD starts program.
Host sends size of header data packet (in bytes) (32-bit int)
Host sends clock frequency over UART (64-bit Hz value)
Host sends test ID (8-bit value)
Host sends header data
Chip responds with BEL (7)
Chip performs work. Host ignores any UART that is not ETB.
Chip responds with ETB (23)
Chip sends size of payload packet (in bytes) (32-bit int)
Chip responds with Payload
Chip awaits new header packet
'''

try:

    # TODO: Replace with class variable.
    elf = '../dspbench/build/dsp24-bmarks/example-bmark/example-bmark.elf'

    # Clock range (MHz) (500M - 1.3G)
    start_clk, end_clk, step_clk = 100, 150, 50

    # Voltage range (V) (Ideal range 0.45 - 1.2V)
    start_voltage, end_voltage, step_v = 0.85, 0.85, 0.05

    smu_init()
    smu_enable()

    cur_clk = start_clk
    cur_v = start_voltage

    ocd = ocd_start()

    # try:
    while cur_clk <= end_clk:
        cur_v = start_voltage
        while cur_v <= end_voltage:

            # # Set the voltage and enable the output.
            smu_set_limits(voltage=cur_v)
            
            # # OCD uploads program
            # # OCD sets PLL MMIO over JTAG
            # # OCD sets baud rate divisor over JTAG for UART
            #ocd.kill()
            #sleep(3)
            #ftdi_reset()
            sleep(3)
            #ocd = ocd_start()
            ocd_upload_pgm(elf, autorun=True, freq_mhz=cur_clk)

            with OpenOcdTclRpc() as openocd:
                for test in config.SHMOO_TESTS:
                    # openocd.run("halt")
                    # uart_txctl = openocd.run("read_memory 0x10020008 32 1")
                    # uart_txctl = openocd.run("read_memory 0x1002000C 32 1")
                    # LOGGER.debug(f'{openocd.run("read_memory 0x10020018 32 1")])

                    # Host sends size of header data packet (in bytes) (32-bit int)
                    data_pkt_size = len(test.vector)
                    ser.write(data_pkt_size.to_bytes(4, byteorder='little'))
                    LOGGER.info(f'Host: Size of header data packet is {data_pkt_size}')
                    
                    # Host sends clock frequency over UART (Hz) (64-bit int)
                    freq_hz = cur_clk * 1_000_000
                    ser.write(freq_hz.to_bytes(8, byteorder='little'))
                    LOGGER.info(f'Host: Clock frequency is {freq_hz} Hz')

                    # Host sends test ID (8-bit value)
                    ser.write(test.id.to_bytes(1, byteorder='little'))
                    LOGGER.info(f'Host: Current Test ID is {test.id}')

                    # Host sends header data
                    ser.write(test.vector)
                    LOGGER.info(f'Host: Sent the following vector payload: {test.vector}')

                    # Chip responds with BEL (7)
                    LOGGER.info(f'Host: Waiting for BEL (7, 0x07) payload acknowledgment...')
                    ser.read_until(b'\x07')
                    LOGGER.info(f'Chip: Sent BEL (7) payload acknowledgment!')
                    smu.write('smub.measure.p(smub.nvbuffer1)')
                    LOGGER.info(f'SMU: Measurement acquired in nvbuffer1!')

                    # Chip performs work. Host ignores any UART that is not ETB.
                    # Chip responds with ETB (23)
                    LOGGER.info(f'Host: Waiting for ETB (23, 0x17) test completion acknowledgment...')
                    ser.read_until(b'\x17')
                    LOGGER.info(f'Chip: Sent ETB (23, 0x17) test completion acknowledgment!')

                    # Chip sends size of payload packet (in bytes) (32-bit int)
                    LOGGER.info(f'Host: Awaiting chip payload packet size...')
                    payload_size_bytes = ser.read(4)
                    payload_size = struct.unpack('<i', payload_size_bytes)[0]
                    LOGGER.info(f'Chip: Payload packet size is {payload_size} ({payload_size_bytes})')

                    # Chip responds with Payload
                    LOGGER.info(f'Host: Waiting for chip payload packet...')
                    chip_payload = ser.read(payload_size)
                    LOGGER.info(f'Chip: Payload packet: {chip_payload}')

                    LOGGER.info(f'Host (Post-Process): Expected packet: {test.expect}')
                    test_passed = test.check_output(chip_payload)
                    LOGGER.info(f'Host (Post-Process): Test result for Test ID {test.id}: {test_passed}')
                
            cur_v += step_v
        cur_clk += step_clk
except Exception as inst:
    print(traceback.format_exc())
ocd.send_signal(11) #segfault, kill with fire
ocd.kill()
ser.close()
#smu_disable()

openocd -f ./platform/dsp24/dsp24.cfg


Open On-Chip Debugger 0.12.0+dev-00772-g40d58ce52 (2024-11-07-12:34)
Licensed under GNU GPL v2
For bug reports, read
	http://openocd.org/doc/doxygen/bugs.html
Info : clock speed 2000 kHz
Error: JTAG scan chain interrogation failed: all zeroes
Error: Check JTAG interface, timings, target power, etc.
Error: Trying to use configured scan chain anyway...
Warn : Bypassing JTAG setup events due to errors
Info : datacount=8 progbufsize=16
Info : Disabling abstract command reads from CSRs.
Info : Disabling abstract command writes to CSRs.
Info : Vector support with vlenb=32
Info : Examined RISC-V core; found 4 harts
Info :  hart 0: XLEN=64, misa=0x800000000034112d
Info : [riscv.cpu0] Examination succeed
Info : [riscv.cpu0] starting gdb server on 3333
Info : Listening on port 3333 for gdb connections
Ready for Remote Connections
Info : Listening on port 6666 for tcl connections
Info : Listening on port 4444 for telnet connections
Info : accepting 'tcl' connection on tcp/6666
0 9912 bytes writte

KeyboardInterrupt: 

In [17]:
data_pkt_size = len(test.vector)
ser.write(data_pkt_size.to_bytes(4, byteorder='little'))
LOGGER.info(f'Host: Size of header data packet is {data_pkt_size}')

# Host sends clock frequency over UART (Hz) (64-bit int)
freq_hz = cur_clk * 1_000_000
ser.write(freq_hz.to_bytes(8, byteorder='little'))
LOGGER.info(f'Host: Clock frequency is {freq_hz} Hz')

# Host sends test ID (8-bit value)
ser.write(test.id.to_bytes(1, byteorder='little'))
LOGGER.info(f'Host: Current Test ID is {test.id}')

# Host sends header data
ser.write(test.vector)
LOGGER.info(f'Host: Sent the following vector payload: {test.vector}')

# Chip responds with BEL (7)
LOGGER.info(f'Host: Waiting for BEL (7, 0x07) payload acknowledgment...')
ser.read_until(b'\x07')
LOGGER.info(f'Chip: Sent BEL (7) payload acknowledgment!')
smu.write('smub.measure.p(smub.nvbuffer1)')


# Chip performs work. Host ignores any UART that is not ETB.
# Chip responds with ETB (23)
LOGGER.info(f'Host: Waiting for ETB (23, 0x17) test completion acknowledgment...')
ser.read_until(b'\x17')
LOGGER.info(f'Chip: Sent ETB (23, 0x17) test completion acknowledgment!')

# Chip sends size of payload packet (in bytes) (32-bit int)
LOGGER.info(f'Host: Awaiting chip payload packet size...')
payload_size_bytes = ser.read(4)
payload_size = struct.unpack('<i', payload_size_bytes)[0]
LOGGER.info(f'Chip: Payload packet size is {payload_size} ({payload_size_bytes})')

# Chip responds with Payload
LOGGER.info(f'Host: Waiting for chip payload packet...')
chip_payload = ser.read(payload_size)
LOGGER.info(f'Chip: Payload packet: {chip_payload}')

LOGGER.info(f'Host (Post-Process): Expected packet: {test.expect}')
test_passed = test.check_output(chip_payload)
LOGGER.info(f'Host (Post-Process): Test result for Test ID {test.id}: {test_passed}')

SerialException: device reports readiness to read but returned no data (device disconnected or multiple access on port?)

In [57]:
smu_disable()

In [7]:
smu_enable()

In [12]:
ser.close()

In [50]:
ftdi_reset()

In [53]:
ocd = ocd_start()
sleep(0.5)
freq_mhz=100
freq_ratio = freq_mhz // pll_ref_clock
with OpenOcdTclRpc() as openocd:
    ocd_cmd(openocd, f'load_image ./example-bmark.elf 0x0 elf')
    ocd_cmd(openocd, 'read_memory 0x130000 32 1')
    ocd_cmd(openocd, 'write_memory 0x130000 32 {0x00000000}')
    
    # PLL Settings
    ratio = int_to_32bit_hex(freq_ratio)
    ocd_cmd(openocd, 'write_memory 0x140060 32 {0x00000000}') # PLLEN = 0;
    ocd_cmd(openocd, 'write_memory 0x140074 32 {0x00000001}') # MDIV_RATIO = 1;
    ocd_cmd(openocd, f'write_memory 0x14006C 32 {{{ratio}}}') # RATIO = ratio;
    ocd_cmd(openocd, 'write_memory 0x140070 32 {0x00000000}') # FRACTION = 0;
    ocd_cmd(openocd, 'write_memory 0x140078 32 {0x00000001}') # ZDIV0_RATIO = 1;
    ocd_cmd(openocd, 'write_memory 0x140080 32 {0x00000001}') # ZDIV1_RATIO = 1;
    ocd_cmd(openocd, 'write_memory 0x140064 32 {0x00000001}') # LDO_ENABLE = 1;
    ocd_cmd(openocd, 'write_memory 0x140060 32 {0x00000001}') # PLLEN = 1;
    ocd_cmd(openocd, 'write_memory 0x14005C 32 {0x00000001}') # POWERGOOD_VNN = 1;
    ocd_cmd(openocd, 'write_memory 0x140100 32 {0x00000001}') # PLLFWEN_B = 1;
    
    ocd_cmd(openocd, 'read_memory 0x140060 32 1') # PLLEN = 0;
    ocd_cmd(openocd, 'read_memory 0x140074 32 1') # MDIV_RATIO = 1;
    ocd_cmd(openocd, 'read_memory 0x14006C 32 1') # RATIO = ratio;
    ocd_cmd(openocd, 'read_memory 0x140070 32 1') # FRACTION = 0;
    ocd_cmd(openocd, 'read_memory 0x140078 32 1') # ZDIV0_RATIO = 1;
    ocd_cmd(openocd, 'read_memory 0x140080 32 1') # ZDIV1_RATIO = 1;
    ocd_cmd(openocd, 'read_memory 0x140064 32 1') # LDO_ENABLE = 1;
    ocd_cmd(openocd, 'read_memory 0x140060 32 1') # PLLEN = 1;
    ocd_cmd(openocd, 'read_memory 0x14005C 32 1') # POWERGOOD_VNN = 1;
    ocd_cmd(openocd, 'read_memory 0x140100 32 1') # PLLFWEN_B = 1;

    # UART Settings
    uart_divisor = ((freq_mhz*1000000) // 115200) - 1
    uart_divisor_hex = int_to_32bit_hex(uart_divisor)
    ocd_cmd(openocd, f'write_memory 0x10020000 32 {{{uart_divisor_hex}}}')
    ocd_cmd(openocd, 'resume 0x80000000')
    sleep(1)
    ocd_cmd(openocd, 'halt')
    ocd_cmd(openocd, 'get_reg pc')
    
ocd.send_signal(11)
ocd.kill()

openocd -f ./platform/dsp24/dsp24.cfg


Open On-Chip Debugger 0.12.0+dev-00772-g40d58ce52 (2024-11-07-12:34)
Licensed under GNU GPL v2
For bug reports, read
	http://openocd.org/doc/doxygen/bugs.html
Info : clock speed 2000 kHz
Info : JTAG tap: riscv.cpu tap/device found: 0x20000913 (mfg: 0x489 (SiFive Inc), part: 0x0000, ver: 0x2)
Info : datacount=8 progbufsize=16
Info : Disabling abstract command reads from CSRs.
Info : Disabling abstract command writes to CSRs.
Info : Vector support with vlenb=32
Info : Examined RISC-V core; found 4 harts
Info :  hart 0: XLEN=64, misa=0x800000000034112d
Info : [riscv.cpu0] Examination succeed
Info : [riscv.cpu0] starting gdb server on 3333
Info : Listening on port 3333 for gdb connections
Ready for Remote Connections
Info : Listening on port 6666 for tcl connections
Info : Listening on port 4444 for telnet connections
Info : accepting 'tcl' connection on tcp/6666
0 9752 bytes written at address 0x80000000
2496 bytes written at address 0x80002618
downloaded 12248 bytes in 0.157314s (76.032 

OpenOCD | load_image ./example-bmark.elf 0x0 elf | 9752 bytes written at address 0x80000000
2496 bytes written at address 0x80002618
downloaded 12248 bytes in 0.157314s (76.032 KiB/s)
OpenOCD | read_memory 0x130000 32 1 | 0x0
OpenOCD | write_memory 0x130000 32 {0x00000000} | 
OpenOCD | write_memory 0x140060 32 {0x00000000} | 
OpenOCD | write_memory 0x140074 32 {0x00000001} | 
OpenOCD | write_memory 0x14006C 32 {0x00000002} | 
OpenOCD | write_memory 0x140070 32 {0x00000000} | 
OpenOCD | write_memory 0x140078 32 {0x00000001} | 
OpenOCD | write_memory 0x140080 32 {0x00000001} | 
OpenOCD | write_memory 0x140064 32 {0x00000001} | 
OpenOCD | write_memory 0x140060 32 {0x00000001} | 
OpenOCD | write_memory 0x14005C 32 {0x00000001} | 
OpenOCD | write_memory 0x140100 32 {0x00000001} | 
OpenOCD | read_memory 0x140060 32 1 | 0x1
OpenOCD | read_memory 0x140074 32 1 | 0x1
OpenOCD | read_memory 0x14006C 32 1 | 0x2
OpenOCD | read_memory 0x140070 32 1 | 0x0
OpenOCD | read_memory 0x140078 32 1 | 0x1
Ope

0 
0 
0 
0 
0 
0 
0 
0 
0 0x1
0 0x1
0 0x2
0 0x0
0 0x1
0 0x1
0 0x1
0 0x1
0 0x1
0 0x1
0 
0 


OpenOCD | write_memory 0x10020000 32 {0x00000363} | 
OpenOCD | resume 0x80000000 | 
OpenOCD | halt | 
OpenOCD | get_reg pc | pc 0x0000000080000f10


0 
0 pc 0x0000000080000f10
Info : dropped 'tcl' connection
